# V38 Step 05 — Recode Model Nonpurchase and Construct Trip Food Spending

**Healthy Purchase Cycle | Milk–Beverage Combined HMM | 2008–2017 | Python + DuckDB**

This notebook follows V38 Step 04. It keeps the five modeled category outcomes, converts category nonpurchase from `NULL` to `0`, and constructs all-food spending independently from raw Nielsen purchase lines and Magnet purchase lines. The combined Beverage spending from Step 04 is **never added to `food_spend`**: beverage, plain milk, and flavored milk purchases enter the denominator only once through the raw purchase-line construction (subject to the food-module list and UPC match).

Milk and original Beverage source fields remain audit fields. `milk_y` and `beverage_y_original` are not modeled outcomes and remain nullable in the corrected panel. Canned milk and buttermilk may remain part of total food spending if included by the pre-existing food-module list, even though they are excluded from the V38 Beverage classification.

Outputs are written to a separate local V38 data directory, outside GitHub. Source data and V37 outputs are not overwritten.

In [1]:
from pathlib import Path
import duckdb, pandas as pd, time, shutil

REPO = Path(r"C:/Users/rwen13/Documents/healthy_purchase_cycle")
DATA = Path(r"C:/Users/rwen13/Documents/healthy_purchase_data")
RAW_HMS = Path(r"C:/Users/rwen13/Documents/2004-2020_Panel_Data/nielsen_extracts/HMS")
FOOD_LIST = REPO / "data_cleaning/upc_classification/food_list.xlsx"
MASTER = RAW_HMS / "Master_Files/Latest/products.tsv"

V38 = DATA / "intermediate/v38_milk_beverage_combined"
SOURCE_PANEL = V38 / "04_category_trip_merge/trip_purchase_panel"
CORRECTED_PANEL = V38 / "05_trip_food_spending/trip_purchase_panel_y0"
FOOD_PANEL = V38 / "05_trip_food_spending/trip_food_spending"
STAGE = V38 / "05_trip_food_spending/stage"
AUDIT = V38 / "05_trip_food_spending/validation"
TEMP = DATA / "temp/v38_milk_beverage_combined/food_spending"
MAGNET_DIR = DATA / "intermediate/magnet_food_purchases"
YEARS = range(2008, 2018)
SOURCE_YEARS = range(2008, 2019)   # 2018 source may contain late-2017 purchases
MEMORY_LIMIT, THREADS = '4GB', 4

# Confirmed all-food spending convention: net of coupon, floored at zero.
# This is computed from source purchase lines once; category spending is not added again.
USE_CAPPED_PRICE = True

for folder in [CORRECTED_PANEL, FOOD_PANEL, STAGE, AUDIT, TEMP]:
    folder.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(database=':memory:')
con.execute(f"SET memory_limit='{MEMORY_LIMIT}'")
con.execute(f"SET threads={THREADS}")
con.execute(f"SET temp_directory='{TEMP.as_posix()}'")

def qpath(p):
    return "'" + Path(p).as_posix().replace("'", "''") + "'"
def qname(s):
    return '"' + s.replace('"','""') + '"'
def parquet_ref(path):
    return f'read_parquet({qpath(path)})'
def raw_ref(path):
    return f"read_csv({qpath(path)}, delim='\t', header=true, all_varchar=true)"

assert FOOD_LIST.is_file(), FOOD_LIST
assert MASTER.is_file(), MASTER
print('Ready: DuckDB', duckdb.__version__)
print('V38 Step 04 input:', SOURCE_PANEL)
print('V38 Step 05 output:', V38 / '05_trip_food_spending')

Ready: DuckDB 1.5.5
V38 Step 04 input: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge\trip_purchase_panel
V38 Step 05 output: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\05_trip_food_spending


## A. Recode model-category nonpurchase outcomes

The V38 Step 04 trip union contains five model outcomes (`fv_y`, combined `beverage_y`, `yogurt_y`, `cereal_y`, and `snack_y`). A `NULL` means that category was not purchased on that trip. This cell writes a separate V38 file with those **five model outcomes** recoded from `NULL` to `0`.

`milk_y` and `beverage_y_original` are retained as nullable audit fields, not recoded or used as separate emissions. Milk spending components and original Beverage spending are also retained as audit fields. Only model-category spending variables are zero-filled for nonpurchase.

In [2]:
MODEL_OUTCOMES = ['fv_y','beverage_y','yogurt_y','cereal_y','snack_y']
AUDIT_OUTCOMES = ['milk_y','beverage_y_original']
MODEL_SPENDS = [
    'fresh_spending','non_fresh_spending',
    'skimmed_yogurt_spending','low_fat_yogurt_spending','other_yogurt_spending',
    'low_sugar_cereal_spending','medium_sugar_cereal_spending','high_sugar_cereal_spending',
    'healthy_snack_spending','unhealthy_snack_spending',
    'healthy_beverage_spending','unhealthy_beverage_spending'
]
AUDIT_SPENDS = [
    'plain_milk_spending','flavored_milk_spending','total_milk_spending',
    'healthy_beverage_spending_original','unhealthy_beverage_spending_original'
]
EXPECTED = {'fv_y':{1,2},'beverage_y':{1,2},'yogurt_y':{1,2},
            'cereal_y':{1,2,3},'snack_y':{1,2}}
recode_audit=[]
for year in YEARS:
    source = SOURCE_PANEL / f'panel_year={year}' / 'trips.parquet'
    assert source.is_file(), f'Run V38 Step 04 first: {source}'
    rel=parquet_ref(source)
    cols = set(con.execute(f'DESCRIBE SELECT * FROM {rel}').df()['column_name'])
    required = set(['hhid','panel_year','trip_id','purchase_date',*MODEL_OUTCOMES,*AUDIT_OUTCOMES,*MODEL_SPENDS,*AUDIT_SPENDS])
    assert required <= cols, f'Missing columns in {year}: {sorted(required - cols)}'
    checks = ', '.join(f"COUNT(*) FILTER (WHERE {qname(y)} IS NOT NULL AND {qname(y)} NOT IN ({','.join(map(str,sorted(EXPECTED[y])))})) AS {qname(y)}" for y in MODEL_OUTCOMES)
    invalid = con.execute(f'SELECT {checks} FROM {rel}').fetchone()
    if any(invalid):
        raise ValueError(f'{year} has unexpected non-null model outcome codes: {dict(zip(MODEL_OUTCOMES, invalid))}')
    nulls = con.execute('SELECT '+ ', '.join(f'COUNT(*) FILTER (WHERE {qname(y)} IS NULL)' for y in MODEL_OUTCOMES)+f' FROM {rel}').fetchone()
    dst= CORRECTED_PANEL / f'panel_year={year}' / 'trips.parquet'
    dst.parent.mkdir(parents=True,exist_ok=True)
    select_expr = []
    schema = con.execute(f'DESCRIBE SELECT * FROM {rel}').df()['column_name'].tolist()
    for field in schema:
        if field in MODEL_OUTCOMES:
            select_expr.append(f'COALESCE({qname(field)}, 0)::INTEGER AS {qname(field)}')
        elif field in MODEL_SPENDS:
            select_expr.append(f'COALESCE({qname(field)}, 0)::DOUBLE AS {qname(field)}')
        else:
            # Keep Milk and source-Beverage audit outcomes/spending unchanged.
            select_expr.append(qname(field))
    con.execute(f"COPY (SELECT {', '.join(select_expr)} FROM {rel}) TO {qpath(dst)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    row={'panel_year':year,'trips':con.execute(f'SELECT COUNT(*) FROM {rel}').fetchone()[0]}
    row.update({f'{y}_null_to_zero':n for y,n in zip(MODEL_OUTCOMES,nulls)})
    row['milk_y_null_audit_rows']=con.execute(f'SELECT COUNT(*) FILTER (WHERE milk_y IS NULL) FROM {rel}').fetchone()[0]
    row['original_beverage_y_null_audit_rows']=con.execute(f'SELECT COUNT(*) FILTER (WHERE beverage_y_original IS NULL) FROM {rel}').fetchone()[0]
    recode_audit.append(row)
pd.DataFrame(recode_audit).to_csv(AUDIT/'model_outcome_zero_recode.csv',index=False)
display(pd.DataFrame(recode_audit))

,panel_year,trips,fv_y_null_to_zero,beverage_y_null_to_zero,yogurt_y_null_to_zero,cereal_y_null_to_zero,snack_y_null_to_zero,milk_y_null_audit_rows,original_beverage_y_null_audit_rows
0,2008,3368393,1593600,1261231,3001486,2936829,1975408,2356375,1810250
1,2009,3699872,1685774,1416650,3278063,3223296,2159255,2602922,2009543
2,2010,4027911,1804812,1566500,3539490,3515224,2338537,2857650,2198881
3,2011,4206761,1840998,1683933,3685404,3699762,2417455,3023806,2332274
4,2012,4154835,1826675,1669380,3648541,3673275,2407662,3045381,2279787
5,2013,4119659,1799994,1678517,3601389,3662514,2382558,3064157,2258895
6,2014,4196848,1801094,1731969,3667676,3744989,2415932,3169217,2295050
7,2015,4145029,1718136,1711120,3620775,3712210,2404246,3150090,2246356
8,2016,4122803,1691638,1701372,3615767,3714182,2392120,3159358,2216568
9,2017,3704316,1478363,1565107,3256464,3366296,2142876,2861654,2021019


## B. Load verified food-module list and build a UPC/module lookup

The complete `food_list.xlsx` continues to define which modules enter the food denominator. UPC and UPC-version remain strings; only module codes are normalized. Conflicting master-product mappings stop the run.

This cell also audits whether plain/flavored milk modules (3625/3592) and excluded-from-beverage canned milk/buttermilk modules (1297/3626) are included in the all-food module list. Any included module contributes to `food_spend` once from source purchase lines; none of the Step 04 category spending is added to the denominator.

In [3]:
food_list=pd.read_excel(FOOD_LIST, dtype={'product_module_code':'string'})
assert 'product_module_code' in food_list.columns
food_list['module']=(food_list['product_module_code'].astype('string').str.strip()
    .str.replace(r'\.0+$','',regex=True).str.replace(r'^0+(?=\d)', '',regex=True))
modules=sorted(set(food_list['module'].dropna())-{'','<NA>'})
assert modules, 'Empty food-module list'
con.register('food_module_df',pd.DataFrame({'module':modules}))
con.execute('CREATE OR REPLACE TEMP TABLE food_modules AS SELECT DISTINCT module FROM food_module_df')
print('Allowed food modules:',len(modules))
master_cols=set(con.execute(f'DESCRIBE SELECT * FROM {raw_ref(MASTER)}').df()['column_name'])
for req in ['upc','upc_ver_uc','product_module_code']:
    assert req in master_cols, f'Master missing {req}'
con.execute(f"""
 CREATE OR REPLACE TEMP TABLE product_module_lookup AS
 SELECT TRIM(upc) AS upc, TRIM(upc_ver_uc) AS upc_ver_uc,
        CASE WHEN ltrim(regexp_replace(TRIM(product_module_code), '\\.0+$',''), '0') = ''
             THEN '0' ELSE ltrim(regexp_replace(TRIM(product_module_code), '\\.0+$',''), '0') END AS module
 FROM {raw_ref(MASTER)}
 WHERE upc IS NOT NULL AND upc_ver_uc IS NOT NULL
""")
conflicts=con.execute('''SELECT COUNT(*) FROM (
 SELECT upc,upc_ver_uc FROM product_module_lookup
 GROUP BY 1,2 HAVING COUNT(DISTINCT module)>1)''').fetchone()[0]
if conflicts:
    raise ValueError(f'Master has {conflicts} UPC-version keys with conflicting modules; resolve before aggregation')
con.execute('''CREATE OR REPLACE TEMP TABLE food_product_lookup AS
 SELECT DISTINCT p.upc,p.upc_ver_uc,p.module FROM product_module_lookup p
 INNER JOIN food_modules m ON p.module=m.module''')
print('Food UPC-versions:',con.execute('SELECT COUNT(*) FROM food_product_lookup').fetchone()[0])

# Check denominator coverage for the milk modules used by V38 plus legacy dairy
# modules. Exclusion from Beverage does not remove a product from total food spend
# when the existing food-module definition includes it.
milk_modules = ['3625','3592','1297','3626']
food_list_coverage = con.execute("""
    SELECT m.module,
           COUNT(DISTINCT p.upc || '|' || p.upc_ver_uc) AS master_upc_versions,
           COUNT(DISTINCT f.upc || '|' || f.upc_ver_uc) AS included_food_upc_versions,
           m.module IN (SELECT module FROM food_modules) AS module_in_food_list
    FROM (SELECT UNNEST(?) AS module) m
    LEFT JOIN product_module_lookup p ON p.module=m.module
    LEFT JOIN food_product_lookup f ON f.upc=p.upc AND f.upc_ver_uc=p.upc_ver_uc
    GROUP BY m.module ORDER BY m.module
""", [milk_modules]).df()
food_list_coverage.to_csv(AUDIT/'milk_module_food_denominator_coverage.csv',index=False)
display(food_list_coverage)

Allowed food modules: 891
Food UPC-versions: 2355125


,module,master_upc_versions,included_food_upc_versions,module_in_food_list
0,1297,810,810,True
1,3592,4352,4352,True
2,3625,14041,14041,True
3,3626,1146,1146,True


## C. Stage raw food spending, one source year at a time

Regular food is identified by the verified `food_list.xlsx` module list and original purchase lines. Magnet food comes from the existing extracted Magnet purchase files. Repeated trip/UPC/version keys are retained. Before excluding Magnet keys from the regular-food branch, this notebook verifies that Magnet and original purchase-line multiplicities match for each key.

Milk is **not joined to or added from the Step 04 category panel** here. If modules 3625/3592 occur in `food_list.xlsx`, their source purchase lines enter the all-food denominator through the regular-food branch exactly once. The staging files retain module codes so the notebook can separately audit milk-module dollars already inside `food_spend`. Canned milk and buttermilk remain part of total food expenditure whenever their modules are part of the existing food list.

In [4]:
# Run this inexpensive schema/preflight audit before launching the yearly scans.
for sy in SOURCE_YEARS:
    base=RAW_HMS/str(sy)/'Annual_Files'
    for label,name in [('purchases',f'purchases_{sy}.tsv'),('trips',f'trips_{sy}.tsv')]:
        assert (base/name).is_file(), f'Missing {label}: {base/name}'
    magnet=MAGNET_DIR/f'magnet_food_purchases_{sy}.tsv'
    assert magnet.is_file(), f'Missing extracted magnet file: {magnet}'
print('All source-year purchase, trip and Magnet files found.')
print('V38 trip panel directory:', SOURCE_PANEL)
print('Food module list remains unchanged; inspect milk module coverage audit before interpreting denominator coverage.')
for label, path, needed in [
 ('purchases',RAW_HMS/'2008/Annual_Files/purchases_2008.tsv', ['upc','upc_ver_uc','trip_code_uc','total_price_paid','coupon_value']),
 ('trips',RAW_HMS/'2008/Annual_Files/trips_2008.tsv',['trip_code_uc','household_code','purchase_date']),
 ('magnet',MAGNET_DIR/'magnet_food_purchases_2008.tsv',['upc','upc_ver_uc','trip_code_uc','magnet_food_spend_increment_capped','purchase_date'])]:
    cols=set(con.execute(f'DESCRIBE SELECT * FROM {raw_ref(path)}').df()['column_name'])
    assert set(needed)<=cols, f'{label}: missing {set(needed)-cols}'
    print(label,'columns OK')

All source-year purchase, trip and Magnet files found.
V38 trip panel directory: C:\Users\rwen13\Documents\healthy_purchase_data\intermediate\v38_milk_beverage_combined\04_category_trip_merge\trip_purchase_panel
Food module list remains unchanged; inspect milk module coverage audit before interpreting denominator coverage.
purchases columns OK
trips columns OK
magnet columns OK


In [5]:
# Each source-year file is processed separately; DuckDB executes file scans on disk.
# Intermediate Parquet contains only the records needed for analysis years.
source_audit=[]
for sy in SOURCE_YEARS:
    started=time.perf_counter()
    base=RAW_HMS/str(sy)/'Annual_Files'
    p=raw_ref(base/f'purchases_{sy}.tsv')
    t=raw_ref(base/f'trips_{sy}.tsv')
    m=raw_ref(MAGNET_DIR/f'magnet_food_purchases_{sy}.tsv')
    stage_dir=STAGE/f'source_year={sy}'
    stage_dir.mkdir(parents=True,exist_ok=True)
    # Treat 2018 source-year boundary dates like all other purchase dates.
    con.execute(f"""CREATE OR REPLACE TEMP VIEW year_trips AS
        SELECT TRIM(trip_code_uc) AS trip_code_uc,
               TRIM(household_code) AS hhid,
               TRY_CAST(purchase_date AS DATE) AS purchase_date
        FROM {t}""")
    dup_trip=con.execute('SELECT COUNT(*) FROM (SELECT trip_code_uc FROM year_trips GROUP BY 1 HAVING COUNT(*)>1)').fetchone()[0]
    if dup_trip: raise ValueError(f'{sy}: {dup_trip} duplicate trip_code_uc in trips source')
    # Materialize Magnet keys and amounts for a relatively small source-year output.
    con.execute(f"""CREATE OR REPLACE TEMP TABLE year_magnet AS
        SELECT TRIM(trip_code_uc) trip_code_uc, TRIM(upc) upc,
               TRIM(upc_ver_uc) upc_ver_uc,
               TRY_CAST(purchase_date AS DATE) purchase_date,
               TRIM(household_code) hhid,
               TRY_CAST(magnet_food_spend_increment_capped AS DOUBLE) amount
        FROM {m}""")
    # Repeated trip/UPC/version combinations are valid purchase lines.
    # Never deduplicate them: amounts from every row are required.
    # Compare purchase-line multiplicities against the raw source instead.
    con.execute("""CREATE OR REPLACE TEMP TABLE year_magnet_keys AS
      SELECT trip_code_uc, upc, upc_ver_uc, COUNT(*) AS magnet_rows
      FROM year_magnet
      GROUP BY 1,2,3""")
    magnet_dups=con.execute("""SELECT COUNT(*) FROM year_magnet_keys
      WHERE magnet_rows > 1""").fetchone()[0]

    # Read original purchase lines with original UPC keys preserved as strings.
    con.execute(f"""CREATE OR REPLACE TEMP VIEW year_purchases AS
      SELECT TRIM(trip_code_uc) trip_code_uc, TRIM(upc) upc,
             TRIM(upc_ver_uc) upc_ver_uc,
             TRY_CAST(total_price_paid AS DOUBLE) total_price_paid,
             COALESCE(TRY_CAST(coupon_value AS DOUBLE),0) coupon_value
      FROM {p}""")

    # SEMI JOIN prevents repeated Magnet keys from multiplying purchase rows.
    # A key-level anti-join is valid only if every Magnet purchase line
    # corresponds to exactly one raw purchase line with the same key.
    con.execute("""CREATE OR REPLACE TEMP TABLE year_raw_magnet_key_counts AS
      SELECT p.trip_code_uc, p.upc, p.upc_ver_uc,
             COUNT(*) AS purchase_rows
      FROM year_purchases p
      SEMI JOIN year_magnet_keys m
        USING (trip_code_uc, upc, upc_ver_uc)
      GROUP BY 1,2,3""")
    overlap=con.execute("""SELECT
        COUNT(*) FILTER (WHERE p.purchase_rows IS NULL) AS unmatched_keys,
        COUNT(*) FILTER (WHERE p.purchase_rows IS NOT NULL
                        AND p.purchase_rows <> m.magnet_rows) AS unequal_keys,
        COALESCE(SUM(m.magnet_rows),0) AS magnet_rows,
        COALESCE(SUM(p.purchase_rows),0) AS matched_raw_rows
      FROM year_magnet_keys m LEFT JOIN year_raw_magnet_key_counts p
        USING (trip_code_uc, upc, upc_ver_uc)""").fetchone()
    unmatched_keys, unequal_keys, magnet_rows, raw_rows = overlap
    print(f'{sy}: repeated Magnet trip/UPC keys={magnet_dups:,}; '
          f'unmatched={unmatched_keys:,}; unequal multiplicities={unequal_keys:,}')
    if unmatched_keys or unequal_keys:
        samples=con.execute("""SELECT m.trip_code_uc,m.upc,m.upc_ver_uc,
                  m.magnet_rows,p.purchase_rows
            FROM year_magnet_keys m LEFT JOIN year_raw_magnet_key_counts p
              USING (trip_code_uc,upc,upc_ver_uc)
            WHERE p.purchase_rows IS NULL OR m.magnet_rows <> p.purchase_rows
            LIMIT 15""").df()
        display(samples)
        raise ValueError(f'{sy}: Magnet/raw purchase-line counts mismatch '
                         f'(unmatched={unmatched_keys}, unequal={unequal_keys}). '
                         'Cannot safely apply key-level anti-join.')
    # All same-key raw rows are accounted for in Magnet; exclude that
    # key from regular foods and add all Magnet increments exactly once.
    ambig = unequal_keys
    # Matched regular-food purchases, excluding corresponding Magnet keys.
    out_regular=stage_dir/'regular.parquet'
    con.execute(f"""COPY (
      SELECT tr.hhid, tr.purchase_date, tr.trip_code_uc, f.module AS product_module_code,
             CAST(GREATEST(COALESCE(p.total_price_paid - p.coupon_value,0),0) AS DOUBLE) AS food_amount
      FROM year_purchases p
      INNER JOIN food_product_lookup f USING (upc, upc_ver_uc)
      INNER JOIN year_trips tr USING (trip_code_uc)
      WHERE tr.hhid IS NOT NULL AND tr.purchase_date BETWEEN DATE '2008-01-01' AND DATE '2017-12-31'
        AND NOT EXISTS (
          SELECT 1 FROM year_magnet_keys m
          WHERE m.trip_code_uc=p.trip_code_uc AND m.upc=p.upc AND m.upc_ver_uc=p.upc_ver_uc
        )
    ) TO {qpath(out_regular)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    out_magnet=stage_dir/'magnet.parquet'
    con.execute(f"""COPY (
      SELECT hhid, purchase_date, trip_code_uc,
             CAST(NULL AS VARCHAR) AS product_module_code,
             CAST(COALESCE(amount,0) AS DOUBLE) AS food_amount
      FROM year_magnet
      WHERE hhid IS NOT NULL AND purchase_date BETWEEN DATE '2008-01-01' AND DATE '2017-12-31'
    ) TO {qpath(out_magnet)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    regular_n=con.execute(f'SELECT COUNT(*) FROM {parquet_ref(out_regular)}').fetchone()[0]
    magnet_n=con.execute(f'SELECT COUNT(*) FROM {parquet_ref(out_magnet)}').fetchone()[0]
    source_audit.append({'source_year':sy,'regular_food_purchase_rows':regular_n,
        'magnet_food_purchase_rows':magnet_n,'duplicate_magnet_keys':magnet_dups,
        'unmatched_magnet_keys':unmatched_keys,
        'unequal_row_count_keys':unequal_keys,'magnet_rows_reconciled':magnet_rows,
        'raw_rows_reconciled':raw_rows,'seconds':round(time.perf_counter()-started,1)})
    print(f'{sy}: regular={regular_n:,}, magnet={magnet_n:,}')
pd.DataFrame(source_audit).to_csv(AUDIT/'food_spending_source_audit.csv',index=False)
display(pd.DataFrame(source_audit))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2008: repeated Magnet trip/UPC keys=874,677; unmatched=0; unequal multiplicities=0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2008: regular=46,438,262, magnet=4,480,556


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2009: repeated Magnet trip/UPC keys=953,768; unmatched=0; unequal multiplicities=0


... (35 outputs left out)

2018: repeated Magnet trip/UPC keys=239,923; unmatched=0; unequal multiplicities=0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2018: regular=169,526, magnet=12,690


,source_year,regular_food_purchase_rows,magnet_food_purchase_rows,duplicate_magnet_keys,unmatched_magnet_keys,unequal_row_count_keys,magnet_rows_reconciled,raw_rows_reconciled,seconds
0,2008,46438262,4480556,874677,0,0,4510411,4510411,35.6
1,2009,46073476,4924316,953768,0,0,4924316,4924316,37.4
2,2010,44912911,5199900,1003710,0,0,5199900,5199900,40.1
3,2011,47129997,5809676,1114275,0,0,5809676,5809676,43.8
4,2012,47262733,3065814,237045,0,0,3065814,3065814,43.1
5,2013,47489893,3111419,219178,0,0,3111419,3111419,43.2
6,2014,48715836,3442975,226244,0,0,3442975,3442975,49.0
7,2015,47950119,3701490,223717,0,0,3701490,3701490,50.1
8,2016,50885277,4067281,244830,0,0,4067281,4067281,54.6
9,2017,49880310,4197064,247678,0,0,4197064,4197064,57.8


## D. Sum food dollars by actual trip; match against the V38 corrected panel

The corrected V38 Step 04 panel is read from the separate V38 directory. Food totals are grouped before joining, so purchase lines cannot multiply trip rows. The `food_spend` value comes only from original regular-food purchase lines and Magnet increments. It does **not** add any category-level spending fields from Step 04.

The notebook reports match rates rather than converting unmatched trips to zero. `milk_module_food_spend` is an audit component already contained within `regular_food_spend` and `food_spend`; it is not added again. Review its coverage with `milk_module_food_denominator_coverage.csv`.

In [6]:
# Build per-year totals from staged source-year files using DuckDB wildcard paths.
# Group before joining: never join purchase rows directly to the trip panel.
regular_files=[STAGE/f'source_year={sy}'/'regular.parquet' for sy in SOURCE_YEARS]
magnet_files=[STAGE/f'source_year={sy}'/'magnet.parquet' for sy in SOURCE_YEARS]
def rel_for(paths):
    return 'read_parquet(['+', '.join(qpath(p) for p in paths)+'])'
regular_rel=rel_for(regular_files)
magnet_rel=rel_for(magnet_files)
food_audit=[]
for year in YEARS:
    t0=time.perf_counter()
    panel=CORRECTED_PANEL/f'panel_year={year}'/'trips.parquet'
    assert panel.is_file()
    # Stage annual totals so the final join is small and deterministic.
    food_totals=STAGE/f'food_trip_totals_{year}.parquet'
    con.execute(f"""COPY (
        SELECT hhid, purchase_date, trip_code_uc,
               SUM(regular_amount) AS regular_food_spend,
               SUM(magnet_amount) AS magnet_food_spend,
               SUM(regular_amount+magnet_amount) AS food_spend,
               SUM(CASE WHEN product_module_code IN ('3625','3592','1297','3626')
                        THEN regular_amount ELSE 0 END) AS milk_module_food_spend
        FROM (
          SELECT hhid,purchase_date,trip_code_uc,food_amount AS regular_amount,0::DOUBLE AS magnet_amount,product_module_code
          FROM {regular_rel} WHERE YEAR(purchase_date)={year}
          UNION ALL
          SELECT hhid,purchase_date,trip_code_uc,0::DOUBLE,food_amount,CAST(NULL AS VARCHAR)
          FROM {magnet_rel} WHERE YEAR(purchase_date)={year}
        ) z GROUP BY 1,2,3
    ) TO {qpath(food_totals)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    f=parquet_ref(food_totals); c=parquet_ref(panel)
    dup=con.execute(f'''SELECT COUNT(*) FROM (SELECT hhid,purchase_date,trip_code_uc FROM {f}
      GROUP BY ALL HAVING COUNT(*)>1)''').fetchone()[0]
    assert dup==0, f'Duplicate food trip keys for {year}'
    dest=FOOD_PANEL/f'panel_year={year}'/'trips.parquet'
    dest.parent.mkdir(parents=True,exist_ok=True)
    con.execute(f"""COPY (
      SELECT c.hhid, c.panel_year, c.trip_id, c.purchase_date,
             f.regular_food_spend, f.magnet_food_spend, f.food_spend,
             f.milk_module_food_spend,
             f.trip_code_uc IS NOT NULL AS food_trip_matched
      FROM {c} c LEFT JOIN {f} f
        ON c.hhid=f.hhid AND c.purchase_date=f.purchase_date
           AND c.trip_id=f.trip_code_uc
    ) TO {qpath(dest)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
    o=parquet_ref(dest)
    counts=con.execute(f'''SELECT COUNT(*) AS trips, COUNT(*) FILTER (WHERE food_trip_matched) AS matched,
      COUNT(*) FILTER (WHERE NOT food_trip_matched) AS unmatched,
      SUM(food_spend) AS food_dollars,
      COUNT(*) FILTER (WHERE food_spend < 0) AS negative_spend
      FROM {o}''').fetchone()
    row=dict(zip(['panel_year','matched','unmatched','food_dollars','negative_spend'],[year,*counts[1:]]))
    row['trips']=counts[0]; row['match_rate']=counts[1]/counts[0] if counts[0] else None
    row['matched_milk_module_food_dollars']=con.execute(f'SELECT SUM(milk_module_food_spend) FROM {o}').fetchone()[0]
    row['seconds']=round(time.perf_counter()-t0,1)
    food_audit.append(row)
    print(f'{year}: {counts[1]:,}/{counts[0]:,} category trips matched to food source; rate {row["match_rate"]:.1%}')
pd.DataFrame(food_audit).to_csv(AUDIT/'trip_food_spending_merge_audit.csv',index=False)
display(pd.DataFrame(food_audit))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2008: 3,368,393/3,368,393 category trips matched to food source; rate 100.0%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2009: 3,699,872/3,699,872 category trips matched to food source; rate 100.0%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2010: 4,027,911/4,027,911 category trips matched to food source; rate 100.0%


... (11 outputs left out)

2016: 4,122,803/4,122,803 category trips matched to food source; rate 100.0%


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

2017: 3,704,316/3,704,316 category trips matched to food source; rate 100.0%


,panel_year,matched,unmatched,food_dollars,negative_spend,trips,match_rate,matched_milk_module_food_dollars,seconds
0,2008,3368393,0,9.461650e+07,0,3368393,1.0,4.041286e+06,18.4
1,2009,3699872,0,1.064976e+08,0,3699872,1.0,3.652871e+06,29.3
2,2010,4027911,0,1.172186e+08,0,4027911,1.0,4.046060e+06,43.1
3,2011,4206761,0,1.290157e+08,0,4206761,1.0,4.492921e+06,31.5
4,2012,4154835,0,1.292963e+08,0,4154835,1.0,4.239071e+06,41.9
5,2013,4119659,0,1.310957e+08,0,4119659,1.0,4.355882e+06,42.6
6,2014,4196848,0,1.381371e+08,0,4196848,1.0,4.224831e+06,44.5
7,2015,4145029,0,1.406503e+08,0,4145029,1.0,3.761232e+06,33.5
8,2016,4122803,0,1.387429e+08,0,4122803,1.0,3.431102e+06,34.7
9,2017,3704316,0,1.246044e+08,0,3704316,1.0,2.931987e+06,36.7


## E. Final safety checks and interpretation

- **Match coverage matters.** Inspect `trip_food_spending_merge_audit.csv` and compare V38 rates against prior validated outputs before Step 06.
- `food_spend` is independent of the five category outcomes. Combined Beverage category spending is not an input to the denominator.
- Food expenditures are counted once from source purchase lines: ordinary food purchases use `max(total_price_paid - coupon_value, 0)`; Magnet increments are added after line-multiplicity reconciliation.
- `milk_module_food_spend` is a subset of `regular_food_spend` and `food_spend`, reported only for audit. Do not add it to `food_spend` again.
- Whether each milk module is included by the unchanged complete food-module list is recorded in `milk_module_food_denominator_coverage.csv`. Canned milk/buttermilk may remain in all-food spending even though they are excluded from the V38 beverage classification.
- This construction targets trips observed in the V38 six-input union (now five modeled outcomes). The separate 2007 fresh-FV initialization is not covered.
- Step 06 will combine the recoded V38 category panel and the separate trip food-spending table using `(hhid, panel_year, trip_id, purchase_date)`.
- All restricted inputs, staging files, audits, and outputs remain in local `healthy_purchase_data`, not GitHub.

In [7]:
# Quick QA: model outcomes and food-spending values
for year in YEARS:
    food=parquet_ref(FOOD_PANEL/f'panel_year={year}'/'trips.parquet')
    panel=parquet_ref(CORRECTED_PANEL/f'panel_year={year}'/'trips.parquet')
    bad=con.execute(f'''SELECT COUNT(*) FROM {food}
        WHERE food_spend<0 OR regular_food_spend<0 OR magnet_food_spend<0
           OR milk_module_food_spend<0''').fetchone()[0]
    assert bad==0, (year,'negative spending',bad)
    missing=con.execute('SELECT '+', '.join(f'COUNT(*) FILTER (WHERE {qname(y)} IS NULL)' for y in MODEL_OUTCOMES)+f' FROM {panel}').fetchone()
    assert not any(missing), (year,'unexpected missing model outcome',missing)
    # Preserve audit NULLs and confirm modeled outcomes use only allowed codes.
    invalid=con.execute(f'''SELECT COUNT(*) FROM {panel} WHERE
        fv_y NOT IN (0,1,2) OR beverage_y NOT IN (0,1,2) OR
        yogurt_y NOT IN (0,1,2) OR cereal_y NOT IN (0,1,2,3) OR
        snack_y NOT IN (0,1,2)''').fetchone()[0]
    assert invalid==0, (year,'invalid recoded model outcome',invalid)
print('Model outcomes and food-spending checks passed. Review match-rate and milk-module coverage audits.')

Model outcomes and food-spending checks passed. Review match-rate and milk-module coverage audits.
